In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.env_checker import check_env
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
import matplotlib.pyplot as plt


In [ ]:
tickers = ["AAPL", "MSFT", "TSLA", "GOOGL", "AMZN"]

data = yf.download(
    tickers,
    start="2020-01-01",
    end="2022-12-31",
    interval="1d"
)

# Séparer prix et volume
price_df = data["Close"].copy()
volume_df = data["Volume"].copy()

print(price_df.head())
print(volume_df.head())


In [ ]:
def prepare_features(price_df: pd.DataFrame, volume_df: pd.DataFrame):
    tickers = list(price_df.columns)
    feats = {}

    for t in tickers:
        px = price_df[t].copy()
        vol = volume_df[t].copy()

        df_t = pd.DataFrame(index=price_df.index)
        df_t["close"] = px

        # Tout est décalé d'un jour pour que la décision à t
        # n'utilise que l'information disponible avant l'ouverture de t
        df_t["ret_1"] = px.pct_change(1).shift(1)
        df_t["ret_5"] = px.pct_change(5).shift(1)
        df_t["vol_20"] = px.pct_change().rolling(20).std().shift(1) * np.sqrt(252)

        vol_mean = vol.rolling(20).mean().shift(1)
        vol_std = vol.rolling(20).std().shift(1)
        df_t["vol_z"] = (vol.shift(1) - vol_mean) / (vol_std + 1e-8)

        feats[t] = df_t

    panel = pd.concat(feats, axis=1)
    panel = panel.dropna().copy()
    return panel

In [ ]:
prepare_features(price_df, volume_df).head()

In [ ]:
class MultiAssetTradingEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(
        self,
        features_panel,
        tickers,
        initial_cash=100000.0,
        transaction_cost=0.001,
        short_borrow_rate=0.03,   # coût annuel du short
        max_gross_exposure=1.0,   # somme(|poids|) max
        max_steps=252,
        random_start=True,
    ):
        super().__init__()

        self.data = features_panel.copy()
        self.tickers = list(tickers)
        self.n_assets = len(self.tickers)

        self.initial_cash = float(initial_cash)
        self.transaction_cost = float(transaction_cost)
        self.short_borrow_rate = float(short_borrow_rate)
        self.max_gross_exposure = float(max_gross_exposure)
        self.max_steps = int(max_steps)
        self.random_start = bool(random_start)

        # Features par actif
        self.asset_features = ["ret_1", "ret_5", "vol_20", "vol_z"]
        self.n_features_per_asset = len(self.asset_features)

        # Observation = features multi-actifs + poids actifs + poids cash
        self.obs_dim = self.n_assets * self.n_features_per_asset + self.n_assets + 1

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.obs_dim,),
            dtype=np.float32,
        )

        # Action continue :
        # - n_assets valeurs signées -> direction / intensité long-short
        # - 1 valeur -> budget d'exposition brute total
        self.action_space = spaces.Box(
            low=-5.0,
            high=5.0,
            shape=(self.n_assets + 1,),
            dtype=np.float32,
        )

        self.dates = self.data.index.unique().tolist()
        self._reset_internal_state()

    def _reset_internal_state(self):
        self.current_step = 0
        self.start_step = 0
        self.end_step = min(len(self.dates) - 2, self.max_steps)

        self.cash = self.initial_cash
        self.cash_weight = 1.0
        self.weights = np.zeros(self.n_assets, dtype=np.float64)
        self.portfolio_value = self.initial_cash
        self.portfolio_values = [self.initial_cash]
        self.n_trades = 0

    @staticmethod
    def _sigmoid(x):
        return 1.0 / (1.0 + np.exp(-x))

    def _action_to_target_weights(self, action):
        action = np.asarray(action, dtype=np.float64).reshape(-1)

        if action.shape[0] != self.n_assets + 1:
            raise ValueError(
                f"Action de taille {action.shape[0]} reçue, attendu {self.n_assets + 1}"
            )

        # Partie 1 : score signé par actif
        signed_scores = np.tanh(np.clip(action[:self.n_assets], -5.0, 5.0))

        # Partie 2 : budget d'exposition brute entre 0 et max_gross_exposure
        gross_budget = (
            self._sigmoid(np.clip(action[-1], -5.0, 5.0)) * self.max_gross_exposure
        )

        l1_norm = np.abs(signed_scores).sum()

        if l1_norm < 1e-12 or gross_budget < 1e-12:
            risky_weights = np.zeros(self.n_assets, dtype=np.float64)
        else:
            risky_weights = signed_scores / l1_norm
            risky_weights = risky_weights * gross_budget

        # cash résiduel
        cash_weight = 1.0 - np.abs(risky_weights).sum()
        cash_weight = max(float(cash_weight), 0.0)

        return risky_weights, cash_weight

    def _get_prices(self, step):
        date = self.dates[step]
        return np.array(
            [self.data.loc[date, (t, "close")] for t in self.tickers],
            dtype=np.float64,
        )

    def _get_feature_vector(self, step):
        date = self.dates[step]
        feats = []

        for t in self.tickers:
            for f in self.asset_features:
                feats.append(self.data.loc[date, (t, f)])

        feats = np.array(feats, dtype=np.float32)

        obs = np.concatenate(
            [
                feats,
                self.weights.astype(np.float32),
                np.array([self.cash_weight], dtype=np.float32),
            ],
            axis=0,
        )
        return obs

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._reset_internal_state()

        max_start = max(0, len(self.dates) - self.max_steps - 1)
        if self.random_start and max_start > 0:
            self.start_step = int(self.np_random.integers(0, max_start + 1))
        else:
            self.start_step = 0

        self.current_step = self.start_step
        self.end_step = min(len(self.dates) - 2, self.current_step + self.max_steps)

        self.portfolio_value = self.initial_cash
        self.weights = np.zeros(self.n_assets, dtype=np.float64)
        self.cash_weight = 1.0
        self.cash = self.initial_cash
        self.portfolio_values = [self.initial_cash]
        self.n_trades = 0

        obs = self._get_feature_vector(self.current_step)
        info = {"portfolio_value": self.portfolio_value}
        return obs, info

    def step(self, action):
        target_weights, target_cash_weight = self._action_to_target_weights(action)

        prices_t = self._get_prices(self.current_step)
        old_value = self.portfolio_value

        prev_allocation = np.concatenate([self.weights, [self.cash_weight]])
        target_allocation = np.concatenate([target_weights, [target_cash_weight]])

        turnover = 0.5 * np.abs(target_allocation - prev_allocation).sum()
        trading_cost = turnover * self.transaction_cost * old_value

        # Passage au jour suivant
        self.current_step += 1
        prices_tp1 = self._get_prices(self.current_step)

        asset_returns = (prices_tp1 / prices_t) - 1.0

        # Rendement portefeuille long/short
        gross_portfolio_return = float(np.dot(target_weights, asset_returns))

        # Coût d'emprunt journalier sur la partie short
        short_exposure = float(np.abs(np.minimum(target_weights, 0.0)).sum())
        borrow_cost = old_value * short_exposure * (self.short_borrow_rate / 252.0)

        new_value = old_value * (1.0 + gross_portfolio_return) - trading_cost - borrow_cost
        new_value = max(new_value, 1e-8)

        self.weights = target_weights
        self.cash_weight = target_cash_weight
        self.cash = new_value * self.cash_weight
        self.portfolio_value = new_value
        self.portfolio_values.append(new_value)

        if turnover > 1e-8:
            self.n_trades += 1

        reward = float(np.log(new_value / max(old_value, 1e-8)))

        terminated = bool(new_value <= self.initial_cash * 0.5)
        truncated = bool(self.current_step >= self.end_step)

        obs = self._get_feature_vector(self.current_step)
        info = {
            "portfolio_value": float(new_value),
            "daily_return_gross": float(gross_portfolio_return),
            "daily_return_net": float(new_value / max(old_value, 1e-8) - 1.0),
            "turnover": float(turnover),
            "short_exposure": float(short_exposure),
            "gross_exposure": float(np.abs(target_weights).sum()),
            "net_exposure": float(target_weights.sum()),
            "n_trades": int(self.n_trades),
            "weights": self.weights.copy(),
            "cash_weight": float(self.cash_weight),
            "asset_returns": asset_returns.copy(),
            "date": self.dates[self.current_step],
        }
        return obs, reward, terminated, truncated, info

    def render(self):
        print(
            f"Step={self.current_step}, "
            f"Value={self.portfolio_value:.2f}, "
            f"CashW={self.cash_weight:.2%}, "
            f"GrossExp={np.abs(self.weights).sum():.2%}, "
            f"NetExp={self.weights.sum():.2%}"
        )

In [ ]:
panel = prepare_features(price_df[tickers], volume_df[tickers])

env = MultiAssetTradingEnv(
    features_panel=panel,
    tickers=tickers,
    initial_cash=100000,
    transaction_cost=0.001,
    short_borrow_rate=0.03,
    max_gross_exposure=1.0,
    max_steps=252,
    random_start=True,
)

check_env(env, warn=True)
print("Environnement long/short valide pour SB3.")

In [ ]:
def make_env():
    return MultiAssetTradingEnv(
        features_panel=panel,
        tickers=tickers,
        initial_cash=100000,
        transaction_cost=0.001,
        short_borrow_rate=0.03,
        max_gross_exposure=1.0,
        max_steps=252,
        random_start=True,
    )

train_env = DummyVecEnv([make_env])
train_env = VecMonitor(train_env)

model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.001,
    vf_coef=0.5,
    verbose=1,
    seed=42,
)

model.learn(total_timesteps=200_000)

In [ ]:
model.save("ppo_multiasset_longshort")

In [ ]:
model = PPO.load("ppo_multiasset_longshort")


In [ ]:
def equal_weight_buy_and_hold(price_df, tickers):
    px = price_df[tickers].copy().dropna()
    daily_returns = px.pct_change().dropna()
    ew_returns = daily_returns.mean(axis=1)


    total_return = (1 + ew_returns).prod() - 1
    sharpe = 0.0
    if ew_returns.std() > 1e-12:
        sharpe = ew_returns.mean() / ew_returns.std() * np.sqrt(252)

    equity = (1 + ew_returns).cumprod()
    dd = equity / equity.cummax() - 1
    max_drawdown = dd.min()

    return {
        "total_return": float(total_return),
        "sharpe": float(sharpe),
        "max_drawdown": float(max_drawdown)
    }

baseline = equal_weight_buy_and_hold(price_df.loc[panel.index], tickers)
print(baseline)

In [ ]:
def evaluate_trained_model(model, env, deterministic=True):
    obs, info = env.reset()
    done = False

    portfolio_values = [info["portfolio_value"]]
    dates = [env.dates[env.current_step]]
    rewards = []

    while not done:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        rewards.append(reward)
        portfolio_values.append(info["portfolio_value"])
        dates.append(env.dates[env.current_step])

    values = np.array(portfolio_values, dtype=np.float64)
    returns = pd.Series(values).pct_change().dropna()

    total_return = values[-1] / values[0] - 1.0

    sharpe = 0.0
    if returns.std() > 1e-12:
        sharpe = returns.mean() / returns.std() * np.sqrt(252)

    running_max = np.maximum.accumulate(values)
    drawdowns = values / running_max - 1.0
    max_drawdown = drawdowns.min()

    results = {
        "total_return": float(total_return),
        "sharpe": float(sharpe),
        "max_drawdown": float(max_drawdown),
        "final_portfolio_value": float(values[-1]),
        "avg_reward": float(np.mean(rewards)) if len(rewards) > 0 else 0.0,
        "n_steps": len(rewards),
        "n_trades": info.get("n_trades", None),
        "portfolio_values": values,
        "dates": dates,
    }

    return results

## Test du model sur données jamais vue


In [ ]:
tickers1 = ["AAPL", "MSFT", "TSLA", "GOOGL", "AMZN"]

data = yf.download(
    tickers1,
    start="2023-01-01",
    end="2023-07-01",
    interval="1d"
)

# Séparer prix et volume
price_df = data["Close"].copy()
volume_df = data["Volume"].copy()

tickers = tickers1.copy()

print(tickers)
print(price_df.head())
print(volume_df.head())


In [ ]:
panel = prepare_features(price_df[tickers], volume_df[tickers])

In [ ]:
panel = prepare_features(price_df[tickers], volume_df[tickers])

test_env = MultiAssetTradingEnv(
    features_panel=panel,
    tickers=tickers,
    initial_cash=100000,
    transaction_cost=0.001,
    short_borrow_rate=0.03,
    max_gross_exposure=1.0,
    max_steps=252,
    random_start=False,
)

results = evaluate_trained_model(model, test_env)

print("Résultats du modèle long/short :")
print(f"Total Return     : {results['total_return']:.2%}")
print(f"Sharpe Ratio     : {results['sharpe']:.3f}")
print(f"Max Drawdown     : {results['max_drawdown']:.2%}")
print(f"Valeur finale    : {results['final_portfolio_value']:.2f}")
print(f"Nombre de trades : {results['n_trades']}")

In [ ]:
baseline = equal_weight_buy_and_hold(price_df.loc[panel.index], tickers1)

print("Baseline :")
print(f"Total Return : {baseline['total_return']:.2%}")
print(f"Sharpe       : {baseline['sharpe']:.3f}")
print(f"Max DD       : {baseline['max_drawdown']:.2%}")

In [ ]:
print("Comparaison IA vs baseline")
print(f"Surperformance return : {results['total_return'] - baseline['total_return']:.2%}")
print(f"Différence Sharpe     : {results['sharpe'] - baseline['sharpe']:.3f}")

In [ ]:
def get_ppo_portfolio_curve(model, env, deterministic=True):
    obs, info = env.reset()
    done = False

    portfolio_values = [info["portfolio_value"]]
    dates = [env.dates[env.current_step]]

    while not done:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        portfolio_values.append(info["portfolio_value"])
        dates.append(env.dates[env.current_step])

    curve = pd.DataFrame({
        "date": dates,
        "ppo_portfolio": portfolio_values,
    }).set_index("date")

    return curve

In [ ]:
def get_buy_and_hold_curve(price_df, tickers, initial_cash=100000):
    px = price_df[tickers].copy().dropna()

    # Prix initiaux
    first_prices = px.iloc[0]

    # Capital réparti également
    alloc_per_asset = initial_cash / len(tickers)

    # Nombre de parts achetées au départ
    shares = alloc_per_asset / first_prices

    # Valeur du portefeuille au fil du temps
    portfolio_values = (px * shares).sum(axis=1)

    curve = pd.DataFrame({
        "buy_hold_portfolio": portfolio_values
    })

    return curve

In [ ]:
test_prices = price_df.loc[panel.index, tickers1].copy()

In [ ]:
ppo_curve = get_ppo_portfolio_curve(model, test_env)
bh_curve = get_buy_and_hold_curve(test_prices, tickers1, initial_cash=100000)

In [ ]:
comparison = ppo_curve.join(bh_curve, how="inner")
print(comparison.head())

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(comparison.index, comparison["ppo_portfolio"], label="PPO")
plt.plot(comparison.index, comparison["buy_hold_portfolio"], label="Buy and Hold")
plt.title("Valeur du portefeuille au fil du temps")
plt.xlabel("Date")
plt.ylabel("Valeur du portefeuille")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(comparison.index, np.log(comparison["ppo_portfolio"]), label="PPO")
plt.plot(comparison.index, np.log(comparison["buy_hold_portfolio"]), label="Buy and Hold")
plt.title("Valeur du portefeuille au fil du temps")
plt.xlabel("Date")
plt.ylabel("ln (Valeur du portefeuille)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
comparison_norm = comparison / comparison.iloc[0] * 100

plt.figure(figsize=(12, 6))
plt.plot(comparison_norm.index, comparison_norm["ppo_portfolio"], label="PPO Walk-Forward")
plt.plot(comparison_norm.index, comparison_norm["buy_hold_portfolio"], label="Buy and Hold")
plt.title("Performance normalisée (base 100)")
plt.xlabel("Date")
plt.ylabel("Base 100")
plt.legend()
plt.grid(True)
plt.show()


# Mise en place d'un entrainement de 5 ans tout les 6 mois pendant 5ans

In [ ]:
tickers = ["AAPL", "MSFT", "TSLA", "GOOGL", "AMZN"]

data = yf.download(
    tickers,
    start="2014-01-01",
    end="2025-12-31",
    interval="1d"
)

# Séparer prix et volume
price_df = data["Close"].copy()
volume_df = data["Volume"].copy()

print(price_df.head())
print(volume_df.head())


In [ ]:
def build_walk_forward_windows(first_train_start="2014-01-01", n_windows=10, train_years=5, test_months=6):
    windows = []
    current_train_start = pd.Timestamp(first_train_start)

    for window_id in range(n_windows):
        train_start = current_train_start
        train_end = train_start + pd.DateOffset(years=train_years) - pd.DateOffset(days=1)
        test_start = train_end + pd.DateOffset(days=1)
        test_end = test_start + pd.DateOffset(months=test_months) - pd.DateOffset(days=1)

        windows.append({
            "window_id": window_id,
            "train_start": train_start,
            "train_end": train_end,
            "test_start": test_start,
            "test_end": test_end,
        })

        current_train_start = current_train_start + pd.DateOffset(months=test_months)

    return windows

windows = build_walk_forward_windows(
    first_train_start="2014-01-01",
    n_windows=10,
    train_years=5,
    test_months=6,
)

In [ ]:
for w in windows:
    deb_ent = w["train_start"].strftime("%Y-%m-%d")
    fint_ent = w["train_end"].strftime("%Y-%m-%d")

    price_df_per = price_df.loc[deb_ent:fint_ent]
    volume_df_per = volume_df.loc[deb_ent:fint_ent]
    panel = prepare_features(price_df_per[tickers], volume_df_per[tickers])

    env = MultiAssetTradingEnv(
        features_panel=panel,
        tickers=tickers,
        initial_cash=100000,
        transaction_cost=0.001,
        short_borrow_rate=0.03,
        max_gross_exposure=1.0,
        max_steps=252,
        random_start=True,
    )

    check_env(env, warn=True)
    print(f"Environnement valide pour SB3. Entrainement {w['window_id']}")

    def make_env(panel=panel):
        return MultiAssetTradingEnv(
            features_panel=panel,
            tickers=tickers,
            initial_cash=100000,
            transaction_cost=0.001,
            short_borrow_rate=0.03,
            max_gross_exposure=1.0,
            max_steps=252,
            random_start=True,
        )

    train_env = DummyVecEnv([make_env])
    train_env = VecMonitor(train_env)

    print(f"Entrainement du modèle {w['window_id']} de {deb_ent} -> {fint_ent}")

    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.001,
        vf_coef=0.5,
        verbose=1,
        seed=42,
    )

    model.learn(total_timesteps=200_000)
    model.save(f"ppo_multiasset_longshort_periode{w['window_id']}")

In [ ]:
all_curves = []
all_metrics = []

cash = 100000

for w in windows:
    deb_test = w["test_start"].strftime("%Y-%m-%d")
    fin_test = w["test_end"].strftime("%Y-%m-%d")

    price_df_per = price_df.loc[deb_test:fin_test]
    volume_df_per = volume_df.loc[deb_test:fin_test]

    model = PPO.load(f"ppo_multiasset_longshort_periode{w['window_id']}")
    panel = prepare_features(price_df_per[tickers], volume_df_per[tickers])

    test_env = MultiAssetTradingEnv(
        features_panel=panel,
        tickers=tickers,
        initial_cash=cash,
        transaction_cost=0.001,
        short_borrow_rate=0.03,
        max_gross_exposure=1.0,
        max_steps=252,
        random_start=False,
    )

    results = evaluate_trained_model(model, test_env)

    print(f"Résultats du modèle {w['window_id']} de {deb_test} -> {fin_test}:")
    print(f"Total Return     : {results['total_return']:.2%}")
    print(f"Sharpe Ratio     : {results['sharpe']:.3f}")
    print(f"Max Drawdown     : {results['max_drawdown']:.2%}")
    print(f"Valeur finale    : {results['final_portfolio_value']:.2f}")
    print(f"Nombre de trades : {results['n_trades']}")
    print()

    curve_df = pd.DataFrame({
        "date": results["dates"],
        "ppo_portfolio": results["portfolio_values"],
    }).drop_duplicates(subset="date").set_index("date")

    curve_df["window_id"] = w["window_id"]
    all_curves.append(curve_df)

    all_metrics.append({
        "window_id": w["window_id"],
        "start": deb_test,
        "end": fin_test,
        "total_return": results["total_return"],
        "sharpe": results["sharpe"],
        "max_drawdown": results["max_drawdown"],
        "final_value": results["final_portfolio_value"],
        "n_trades": results["n_trades"],
    })

    cash = results["final_portfolio_value"]

ppo_curve = pd.concat(all_curves).sort_index()
ppo_curve = ppo_curve[~ppo_curve.index.duplicated(keep="first")]

metrics_df = pd.DataFrame(all_metrics)
print(metrics_df)

In [ ]:
baseline_tickers = tickers.copy()


In [ ]:
def get_buy_and_hold_curve2(price_df, tickers, start_date, end_date, initial_cash=100000):
    px = price_df.loc[start_date:end_date, tickers].copy().dropna()

    # achat au début, puis on garde
    first_prices = px.iloc[0]
    alloc_per_asset = initial_cash / len(tickers)
    shares = alloc_per_asset / first_prices

    portfolio_values = (px * shares).sum(axis=1)

    curve = pd.DataFrame({
        "buy_hold_portfolio": portfolio_values
    })

    return curve

In [ ]:
start_date = ppo_curve.index.min()
end_date = ppo_curve.index.max()

baseline_tickers = tickers.copy()

bh_curve = get_buy_and_hold_curve2(
    price_df=price_df,
    tickers=baseline_tickers,
    start_date=start_date,
    end_date=end_date,
    initial_cash=100000,
)


In [ ]:
comparison = ppo_curve[["ppo_portfolio"]].join(bh_curve, how="inner")
print(comparison.head())

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(comparison.index, comparison["ppo_portfolio"], label="PPO Walk-Forward")
plt.plot(comparison.index, comparison["buy_hold_portfolio"], label="Buy and Hold")
plt.title("Valeur du portefeuille : PPO vs Buy and Hold")
plt.xlabel("Date")
plt.ylabel("Valeur du portefeuille")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(comparison.index, np.log(comparison["ppo_portfolio"]), label="PPO Walk-Forward")
plt.plot(comparison.index, np.log(comparison["buy_hold_portfolio"]), label="Buy and Hold")
plt.title("Valeur du portefeuille : PPO vs Buy and Hold")
plt.xlabel("Date")
plt.ylabel(" ln (Valeur du portefeuille)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
comparison_norm = comparison / comparison.iloc[0] * 100

plt.figure(figsize=(12, 6))
plt.plot(comparison_norm.index, comparison_norm["ppo_portfolio"], label="PPO Walk-Forward")
plt.plot(comparison_norm.index, comparison_norm["buy_hold_portfolio"], label="Buy and Hold")
plt.title("Performance normalisée (base 100)")
plt.xlabel("Date")
plt.ylabel("Base 100")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def compute_curve_metrics(curve: pd.Series):
    returns = curve.pct_change().dropna()

    total_return = curve.iloc[-1] / curve.iloc[0] - 1.0

    sharpe = 0.0
    if returns.std() > 1e-12:
        sharpe = returns.mean() / returns.std() * np.sqrt(252)

    running_max = curve.cummax()
    drawdown = curve / running_max - 1.0
    max_drawdown = drawdown.min()

    return {
        "total_return": float(total_return),
        "sharpe": float(sharpe),
        "max_drawdown": float(max_drawdown)
    }

In [ ]:
ppo_metrics_global = compute_curve_metrics(comparison["ppo_portfolio"])
bh_metrics_global = compute_curve_metrics(comparison["buy_hold_portfolio"])

print("=== PPO Walk-Forward ===")
print(f"Total Return : {ppo_metrics_global['total_return']:.2%}")
print(f"Sharpe       : {ppo_metrics_global['sharpe']:.3f}")
print(f"Max DD       : {ppo_metrics_global['max_drawdown']:.2%}")

print("\n=== Buy and Hold ===")
print(f"Total Return : {bh_metrics_global['total_return']:.2%}")
print(f"Sharpe       : {bh_metrics_global['sharpe']:.3f}")
print(f"Max DD       : {bh_metrics_global['max_drawdown']:.2%}")